# ONNX Runtime and CUDA Versions

- ONNX Runtime is built for a specific CUDA version. This means the names of some CUDA libraries are hardcoded inside the package. If the installed libraries have a different version, execution will fail.

- The [tensorrt](https://pypi.org/project/tensorrt/) package is also built for a specific CUDA version. It is possible to find packages for other CUDA versions without rebuilding, such as [tensorrt-cu12](https://pypi.org/project/tensorrt-cu12/), but they install a particular CUDA version into the current environment rather than system-wide, and ONNX Runtime might not see it.


In [ ]:
#!pip install onnxruntime-gpu tensorrt

## Check the CUDA Version


In [5]:
import os, re, site, subprocess
from pathlib import Path
import onnxruntime as ort, tensorrt

def read_cmd(*cmd):
    result = subprocess.run(cmd, capture_output=True, text=True)
    return (result.stdout or result.stderr).strip()

def find_cudart_with_ldd(lib_path):
    if not os.path.exists(lib_path):
        return "N/A", "Not found"
    for line in read_cmd("ldd", lib_path).splitlines():
        if "libcudart.so" not in line:
            continue
        version = re.search(r"libcudart\.so\.(\d+)", line)
        resolved = line.split("=>", 1)[1].split("(", 1)[0].strip() if "=>" in line else line.strip()
        return (version.group(1), resolved) if version else ("N/A", resolved)
    return "N/A", "Not found"

def find_cudart_in_site_packages():
    matches = sorted(Path(site.getsitepackages()[0]).rglob("libcudart.so.*"))
    if not matches:
        return "N/A", "Not found"
    best = max(matches, key=lambda p: int(re.search(r"(\d+)$", p.name).group(1)))
    return re.search(r"(\d+)$", best.name).group(1), str(best)

smi_text = read_cmd("nvidia-smi")
drv_match = re.search(r"Driver Version:\s*([0-9.]+)", smi_text)
drv = drv_match.group(1) if drv_match else "N/A"
if drv == "N/A":
    drv_text = read_cmd("nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader,nounits")
    drv_match = re.search(r"([0-9]+(?:\.[0-9]+)+)", drv_text)
    drv = drv_match.group(1) if drv_match else "N/A"
cuda_drv_match = re.search(r"CUDA Version:\s*([0-9.]+)", smi_text)
cuda_drv = cuda_drv_match.group(1) if cuda_drv_match else "N/A"

sys_cuda_path = os.path.realpath("/usr/local/cuda") if os.path.exists("/usr/local/cuda") else "Not found"
sys_cuda_match = re.search(r"cuda-([0-9.]+)$", sys_cuda_path)
sys_cuda = sys_cuda_match.group(1) if sys_cuda_match else "N/A"

ort_lib = Path(ort.__file__).resolve().parent / "capi" / "libonnxruntime_providers_cuda.so"
ort_cuda, ort_evidence = find_cudart_with_ldd(str(ort_lib))

trt_binding = Path(site.getsitepackages()[0]) / "tensorrt_bindings" / "tensorrt.so"
trt_cuda, trt_evidence = find_cudart_with_ldd(str(trt_binding))
if trt_evidence == "Not found":
    trt_cuda, trt_evidence = find_cudart_in_site_packages()

print(f"NVIDIA Driver:   {drv} (CUDA {cuda_drv})")
print(f"System Toolkit:  CUDA {sys_cuda} ({sys_cuda_path})")
print(f"ONNX Runtime:    v{ort.__version__} (CUDA {ort_cuda}) [{ort_evidence}]")
print(f"TensorRT:        v{tensorrt.__version__} (CUDA {trt_cuda}) [{trt_evidence}]")


NVIDIA Driver:   580.126.09 (CUDA 13.0)
System Toolkit:  CUDA 12.4 (/usr/local/cuda-12.4)
ONNX Runtime:    v1.23.2 (CUDA 12) [/usr/local/cuda/targets/x86_64-linux/lib/libcudart.so.12]
TensorRT:        v10.16.0.72 (CUDA N/A) [Not found]


# Run Attempt

We installed `onnxruntime-gpu` and `tensorrt`, and we already have an ONNX model on disk:

`model_repo/resnet18_onnx/1/model.onnx`

So the first expectation is that ONNX Runtime should be able to run the model with `TensorrtExecutionProvider`.


In [6]:
import numpy as np
import onnxruntime as ort
#set_tensorrt_ld_library_path()

sess = ort.InferenceSession("model/model.onnx",
                            providers=["TensorrtExecutionProvider"])
dummy = {"INPUT_BATCH_OF_IMAGES": np.random.rand(1, 3, 224, 224).astype(np.float32)}
out = sess.run(None, dummy)

print(f"providers={sess.get_providers()}  output={out[0].shape}")

providers=['TensorrtExecutionProvider', 'CPUExecutionProvider']  output=(1, 1000)


## Which CUDA Runtime Was Actually Loaded?

To answer that, inspect `libcudart.so.*` in `/proc/self/maps` after inference and then check which loaded library directly requested that soname.


In [7]:
import re, site, subprocess
import numpy as np
from pathlib import Path
import onnxruntime as ort, tensorrt

sess = ort.InferenceSession("model/model.onnx",
                            providers=["TensorrtExecutionProvider"])
dummy = {"INPUT_BATCH_OF_IMAGES": np.random.rand(1, 3, 224, 224).astype(np.float32)}
_ = sess.run(None, dummy)

mapped_cudart = sorted({
    line.split()[-1]
    for line in Path("/proc/self/maps").read_text().splitlines()
    if "libcudart.so" in line
})

site_packages = Path(site.getsitepackages()[0])
candidates = [
    Path(ort.__file__).resolve().parent / "capi" / "libonnxruntime_providers_tensorrt.so",
    Path(ort.__file__).resolve().parent / "capi" / "libonnxruntime_providers_cuda.so",
    site_packages / "tensorrt_bindings" / "tensorrt.so",
]

print("providers:", sess.get_providers())
for cudart_path in mapped_cudart:
    soname = re.search(r"libcudart\.so\.\d+", cudart_path).group(0)
    why = []
    for lib in candidates:
        if not lib.exists():
            continue
        readelf_out = subprocess.run(["readelf", "-d", str(lib)], capture_output=True, text=True).stdout
        if soname in readelf_out:
            why.append(lib.name)
    print(f"Loaded: {cudart_path}")
    print("Why:", ", ".join(why) if why else "no direct DT_NEEDED match in checked libraries")


providers: ['TensorrtExecutionProvider', 'CPUExecutionProvider']
Loaded: /usr/local/cuda-12.4/targets/x86_64-linux/lib/libcudart.so.12.4.99
Why: libonnxruntime_providers_tensorrt.so, libonnxruntime_providers_cuda.so


In [ ]:
print("Available providers:", ort.get_available_providers())

# Conclusion

- Before installation, check the system CUDA version and select a compatible ONNX Runtime version.
- If the run fails, check that the installed CUDA and cuDNN libraries are visible to ONNX Runtime. Update `LD_LIBRARY_PATH` if needed.
